### 4.0 Document Loaders

There are many other types of Documents that can be loaded in. You can see all the document loaders available here: https://python.langchain.com/docs/modules/data_connection/document_loaders/

Keep in mind many Loaders are dependent on other libraries, meaning issues in those libraries can end up breaking the Langchain loaders.

#### 4.1 CSV Loader

In [49]:
from langchain.document_loaders import CSVLoader

In [51]:
loader = CSVLoader(r'penguins.csv')
data = loader.load()

In [55]:
print(data[1].page_content)

species: Adelie
island: Torgersen
bill_length_mm: 39.5
bill_depth_mm: 17.4
flipper_length_mm: 186
body_mass_g: 3800
sex: FEMALE


#### 4.2 HTML 

In [56]:
from langchain.document_loaders import BSHTMLLoader

In [57]:
loader = BSHTMLLoader(r'some_website.html')
data = loader.load()
data

[Document(metadata={'source': 'some_website.html', 'title': ''}, page_content='Heading 1')]

#### 4.3 PDF

In [ ]:
# !pip install pypdf

In [58]:
from langchain.document_loaders import PyPDFLoader

In [59]:
loader = PyPDFLoader(r'report.pdf')
pages = loader.load_and_split()

In [60]:
print(pages[0].page_content)

Thisisthefirst linePDF.ThisisthesecondlineinthePDF.ThisisthethirdlineinthePDF.


#### 4.4 Document Tranformations: Split by Character, split by tokens

In [61]:
with open(r'FDR_State_of_Union_1944.txt') as file:
    speech_text = file.read()

In [62]:
from langchain.text_splitter import CharacterTextSplitter

In [64]:
text_splitter = CharacterTextSplitter(separator="\n\n",chunk_size=1000) #1000 is default value
text_splitter

In [67]:
texts = text_splitter.create_documents([speech_text])
print(type(texts))
print('\n')
print(texts[2])

<class 'list'>


page_content='To such suspicious souls—using a polite terminology—I wish to say that Mr. Churchill, and Marshal Stalin, and Generalissimo Chiang Kai-shek are all thoroughly conversant with the provisions of our Constitution. And so is Mr. Hull. And so am I.

Of course we made some commitments. We most certainly committed ourselves to very large and very specific military plans which require the use of all Allied forces to bring about the defeat of our enemies at the earliest possible time.

But there were no secret treaties or political or financial commitments.

The one supreme objective for the future, which we discussed for each Nation individually, and for all the United Nations, can be summed up in one word: Security.

And that means not only physical security which provides safety from attacks by aggressors. It means also economic security, social security, moral security—in a family of Nations.'


In [68]:
type(texts[0])

langchain_core.documents.base.Document

In [ ]:
#!pip install tiktoken

In [69]:
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(chunk_size = 500) #now chunk size is a hard length based on tokens

In [70]:
texts = text_splitter.split_text(speech_text)

In [71]:
texts[0]

'This Nation in the past two years has become an active partner in the world\'s greatest war against human slavery.\n\nWe have joined with like-minded people in order to defend ourselves in a world that has been gravely threatened with gangster rule.\n\nBut I do not think that any of us Americans can be content with mere survival. Sacrifices that we and our allies are making impose upon us all a sacred obligation to see to it that out of this war we and our children will gain something better than mere survival.\n\nWe are united in determination that this war shall not be followed by another interim which leads to new disaster- that we shall not repeat the tragic errors of ostrich isolationism—that we shall not repeat the excesses of the wild twenties when this Nation went for a joy ride on a roller coaster which ended in a tragic crash.\n\nWhen Mr. Hull went to Moscow in October, and when I went to Cairo and Teheran in November, we knew that we were in agreement with our allies in our

#### 4.5 Text Embeddings 

In [72]:
from langchain_openai.embeddings import OpenAIEmbeddings

In [73]:
embeddings = OpenAIEmbeddings(api_key=apikey)

In [74]:
text = "Some normal text to send to OpenAI to be embedded into a N dimensional vector"

In [75]:
embedded_text = embeddings.embed_query(text)

In [77]:
embedded_text[0:9]

[-0.010135051794350147,
 -0.004018573090434074,
 0.0032770722173154354,
 0.02355440892279148,
 0.022773120552301407,
 0.020342445001006126,
 -0.015915142372250557,
 0.004828798584640026,
 -0.039527423679828644]

### 5.0 Vector Store

##### We can save the embeddings into a Vector store - Chroma

In [ ]:
#!pip install langchain_chroma

In [78]:
import chromadb
print(chromadb.__version__)

0.5.23


In [79]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.document_loaders import TextLoader

##### Load the document and split(still recommended even if under the context window)

In [80]:
# load the document and split it into chunks
loader = TextLoader(r'FDR_State_of_Union_1944.txt')
documents = loader.load()

In [81]:
# split it into chunks
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(chunk_size=500)
docs = text_splitter.split_documents(documents)

##### Connect to OpenAI for Embeddings

In [82]:
embedding_function = OpenAIEmbeddings(api_key=apikey)

C:\Users\mindf\AppData\Local\Temp\ipykernel_26768\1802820995.py:1: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedding_function = OpenAIEmbeddings(api_key=apikey)


##### Pass Embeddings and Docs into Chroma

In [83]:
# load it into Chroma
db = Chroma.from_documents(docs, embedding_function,persist_directory=r'speech_embedding_db')

##### Save the new embeddings to the disk

In [84]:
# Helpful to force a save
db.persist()

C:\Users\mindf\AppData\Local\Temp\ipykernel_26768\3220725503.py:2: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()


##### Loading embeddings from the disk

In [85]:
persist_directory=r'speech_embedding_db'
db_connection = Chroma(persist_directory=persist_directory, embedding_function=embedding_function)

C:\Users\mindf\AppData\Local\Temp\ipykernel_26768\961613216.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  db_connection = Chroma(persist_directory=persist_directory, embedding_function=embedding_function)


In [86]:
new_doc = "What did FDR say about the cost of food law?"
docs = db_connection.similarity_search(new_doc)

In [87]:
print(docs[0].page_content)

That is the way to fight and win a war—all out—and not with half-an-eye on the battlefronts abroad and the other eye-and-a-half on personal, selfish, or political interests here at home.

Therefore, in order to concentrate all our energies and resources on winning the war, and to maintain a fair and stable economy at home, I recommend that the Congress adopt:

(1) A realistic tax law—which will tax all unreasonable profits, both individual and corporate, and reduce the ultimate cost of the war to our sons and daughters. The tax bill now under consideration by the Congress does not begin to meet this test.

(2) A continuation of the law for the renegotiation of war contracts—which will prevent exorbitant profits and assure fair prices to the Government. For two long years I have pleaded with the Congress to take undue profits out of war.

(3) A cost of food law—which will enable the Government (a) to place a reasonable floor under the prices the farmer may expect for his production; and

##### Adding a new document

In [88]:
# load the document and split it into chunks
loader = TextLoader(r"Lincoln_State_of_Union_1862.txt")
documents = loader.load()

In [89]:
# split it into chunks
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(chunk_size=500)
docs = text_splitter.split_documents(documents)

Created a chunk of size 608, which is longer than the specified 500
Created a chunk of size 539, which is longer than the specified 500
Created a chunk of size 686, which is longer than the specified 500


In [90]:
# load it into Chroma
db = Chroma.from_documents(docs, embedding_function,persist_directory=persist_directory)

In [91]:
docs = db.similarity_search('slavery')

In [92]:
docs[0].page_content

'As to the second article, I think it would be impracticable to return to bondage the class of persons therein contemplated. Some of them, doubtless, in the property sense belong to loyal owners, and hence provision is made in this article for compensating such. The third article relates to the future of the freed people. It does not oblige, but merely authorizes Congress to aid in colonizing such as may consent. This ought not to be regarded as objectionable on the one hand or on the other, insomuch as it comes to nothing unless by the mutual consent of the people to be deported and the American voters, through their representatives in Congress.\n\nI can not make it better known than it already is that I strongly favor colonization; and yet I wish to say there is an objection urged against free colored persons remaining in the country which is largely imaginary, if not sometimes malicious.\n\nIt is insisted that their presence would injure and displace white labor and white laborers. 

### 5.1 Vector Store Retriever

In [93]:
from langchain.document_loaders import WikipediaLoader
from langchain_chroma import Chroma

In [94]:
embedding_function = OpenAIEmbeddings(api_key=apikey)

In [95]:
db_connection = Chroma(persist_directory=r'mk_ultra',embedding_function=embedding_function)

In [96]:
retriever = db_connection.as_retriever()

In [100]:
search_kwargs = {"score_threshold":0.8,"k":4}
docs = retriever.invoke("this",search_kwargs=search_kwargs)

In [102]:
docs 

[]

### 6.0 Exercise : Vector Stores

###  Data Connections Exercise

#### Ask a Legal Research Assistant Bot about the US Constitution

Let's revisit our first exercise and add offline capability using ChromaDB. Your function should do the following:

* Read the US_Constitution.txt file inside the some_data folder
* Split this into chunks (you choose the size)
* Write this to a ChromaDB Vector Store
* Use Context Compression to return the relevant portion of the document to the question

In [103]:
# Build a sample vectorDB
from langchain.vectorstores import Chroma
from langchain_openai import ChatOpenAI
from langchain.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor 

In [110]:
def us_constitution_helper(question):
    '''
    Takes in a question about the US Constitution and returns the most relevant
    part of the constitution. Notice it may not directly answer the actual question!
    
    Follow the steps below to fill out this function:
    '''

    persist_directory=r"constitution"
    
    # PART ONE:
    # LOAD "/US_Constitution in a Document object
    loader = TextLoader(r"US_Constitution.txt")
    documents = loader.load()
    
    # PART TWO
    # Split the document into chunks (you choose how and what size)
    text_splitter = CharacterTextSplitter.from_tiktoken_encoder(chunk_size=500)
    docs = text_splitter.split_documents(documents)
    
    # PART THREE
    # EMBED THE Documents (now in chunks) to a persisted ChromaDB
    embedding_function = OpenAIEmbeddings(api_key=apikey)
    db = Chroma.from_documents(docs, embedding_function,persist_directory=persist_directory)
    db.persist()

    # PART FOUR
    # Use ChatOpenAI and ContextualCompressionRetriever to return the most
    # relevant part of the documents.

    # results = db.similarity_search("What is the 13th Amendment?")
    # print(results[0].page_content) # NEED TO COMPRESS THESE RESULTS!
    llm = ChatOpenAI(temperature=0.5, api_key=apikey)
    compressor = LLMChainExtractor.from_llm(llm)

    compression_retriever = ContextualCompressionRetriever(base_compressor=compressor, 
                                                           base_retriever=db.as_retriever(search_type='mmr'))

    compressed_docs = compression_retriever.invoke(question)

    return compressed_docs[0].page_content

In [111]:
print(us_constitution_helper("What are Powers of Congress? elaborate"))

Section 8: Powers of Congress
The Congress shall have Power To lay and collect Taxes, Duties, Imposts and Excises, to pay the Debts and provide for the common Defence and general Welfare of the United States; but all Duties, Imposts and Excises shall be uniform throughout the United States;

To borrow Money on the credit of the United States;

To regulate Commerce with foreign Nations, and among the several States, and with the Indian Tribes;

To establish a uniform Rule of Naturalization, and uniform Laws on the subject of Bankruptcies throughout the United States;

To coin Money, regulate the Value thereof, and of foreign Coin, and fix the Standard of Weights and Measures;

To provide for the Punishment of counterfeiting the Securities and current Coin of the United States;

To establish Post Offices and post Roads;

To promote the Progress of Science and useful Arts, by securing for limited Times to Authors and Inventors the exclusive Right to their respective Writings and Discoveri